In [3]:
%pip install -q lightgbm optuna

import os
# 3단계에서 드라이브를 마운트했다면 /content/drive/MyDrive 가 보여야 한다
print('드라이브 연결:', os.path.exists('/content/drive/MyDrive'))

드라이브 연결: True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, glob
zip_list = glob.glob('/content/drive/MyDrive/house/*.zip')
ZIP_PATH = zip_list[0]                        # house 폴더 안의 공시가격 zip
SAVE_DIR = '/content/house'                   # 작업 폴더
MODEL_DIR = '/content/house/models'           # 모델 저장 폴더
sido_list = ['서울특별시', '경기도']
os.makedirs(MODEL_DIR, exist_ok=True)
print(ZIP_PATH)

/content/drive/MyDrive/house/국토교통부_주택 공시가격 정보_20260101.zip


In [5]:
import zipfile
import pandas as pd

use_cols = ['시도', '시군구', '읍면', '동리', '단지코드', '호명', '전용면적', '공시가격']

zip = zipfile.ZipFile(ZIP_PATH)                 # zip 안에 파일이 4개라 이름을 지정해서 연다
print(zip.namelist())
csv_name = [name for name in zip.namelist() if name.endswith('(2026).csv')][0]   # 본 데이터 CSV

parts = []
for chunk in pd.read_csv(zip.open(csv_name), usecols=use_cols, dtype=str, chunksize=1_000_000):
    parts.append(chunk[chunk['시도'].isin(sido_list)])   # 서울·경기만 남기기
df = pd.concat(parts, ignore_index=True)
print(df.shape)
df.head()

['국토교통부_주택 공시가격 정보(2026).csv', '국토교통부_주택 공시가격 정보(2026)_레이아웃.png', '국토교통부_주택 공시가격 정보(2026)_레이아웃.xlsx', '국토교통부_주택 공시가격 정보(2026)_샘플데이터.csv']
(7224831, 8)


,시도,시군구,읍면,동리,호명,전용면적,공시가격,단지코드
0,서울특별시,종로구,NaN,청운동,111,187.49,990000000,3
1,서울특별시,종로구,NaN,청운동,112,187.49,990000000,3
2,서울특별시,종로구,NaN,청운동,121,157.25,918000000,3
3,서울특별시,종로구,NaN,청운동,122,157.25,918000000,3
4,서울특별시,종로구,NaN,청운동,131,152.33,896000000,3


In [6]:
print( df.shape )
df.info()
print( df.isnull().sum() )

df['전용면적'] = pd.to_numeric( df['전용면적'] )
df['공시가격'] = pd.to_numeric( df['공시가격'] )

df['법정동'] = df['읍면'].fillna('') + ' ' + df['동리']
df['법정동'] = df['법정동'].str.strip()      

df['호명'] = df['호명'].str.strip()
df = df[ df['호명'].str.isdigit() ]                 
df = df[ df['호명'].str.len().isin( [3, 4] ) ]       
df['층'] = df['호명'].astype(int) // 100

df = df[ df['공시가격'] > 0 ]
df = df[ df['전용면적'] > 0 ]
df = df[ df['층'] > 0 ]

print( '중복 행 수 : ' , df.shape[0] - df.drop_duplicates( subset=['단지코드', '전용면적', '층'] ).shape[0] )
df = df.drop_duplicates( subset=['단지코드', '전용면적', '층'] ).reset_index( drop=True )

print( df.shape )
print( df['시도'].value_counts() )
print( df[ ['전용면적', '층', '공시가격'] ].describe() )
df.head()

(7224831, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7224831 entries, 0 to 7224830
Data columns (total 8 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   시도      object
 1   시군구     object
 2   읍면      object
 3   동리      object
 4   호명      object
 5   전용면적    object
 6   공시가격    object
 7   단지코드    object
dtypes: object(8)
memory usage: 441.0+ MB
시도            0
시군구           0
읍면      6621735
동리            0
호명            0
전용면적          0
공시가격          0
단지코드          0
dtype: int64
중복 행 수 :  4980962
(1951221, 10)
시도
서울특별시    1012344
경기도       938877
Name: count, dtype: int64
               전용면적             층          공시가격
count  1.951221e+06  1.951221e+06  1.951221e+06
mean   6.159291e+01  6.087098e+00  3.341274e+08
std    3.084324e+01  6.058777e+00  5.022172e+08
min    4.180000e+00  1.000000e+00  3.510000e+06
25%    4.077000e+01  2.000000e+00  1.160000e+08
50%    5.886000e+01  4.000000e+00  1.930000e+08
75%    7.890000e+01  7.000000e+00  3.480000e+08
max    5.5289

,시도,시군구,읍면,동리,호명,전용면적,공시가격,단지코드,법정동,층
0,서울특별시,종로구,NaN,청운동,111,187.49,990000000,3,청운동,1
1,서울특별시,종로구,NaN,청운동,121,157.25,918000000,3,청운동,1
2,서울특별시,종로구,NaN,청운동,131,152.33,896000000,3,청운동,1
3,서울특별시,종로구,NaN,청운동,211,187.49,990000000,3,청운동,2
4,서울특별시,종로구,NaN,청운동,221,157.25,918000000,3,청운동,2


In [13]:
sido = '경기도'
df_sido = df[ df['시도'] == sido ].sample(n=150_000, random_state=42)

num_cols = ['전용면적' , '층']
cat_cols = ['시군구' , '법정동']

for col in cat_cols:
    df_sido[col] = df_sido[col].astype('category')
X = df_sido[num_cols + cat_cols]

import numpy as np

Y = np.log1p( df_sido['공시가격'] )
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

import optuna
from lightgbm import LGBMRegressor, early_stopping

optuna.logging.set_verbosity( optuna.logging.WARNING )

# 목적 함수
def objective(trial) :
    from sklearn.model_selection import KFold
    cv = KFold( n_splits=3, shuffle=True, random_state=42 )
    cv_r2_scores = [ ]      # 각 폴드별 검증 점수 저장
    best_iterations = [ ]   # 각 폴드별 치적 트리 번호 저장
    for train_idx, val_idx in cv.split( X_train, Y_train) :
        X_tr, X_val = X_train.iloc[train_idx] , X_train.iloc[val_idx]
        Y_tr, Y_val = Y_train.iloc[ train_idx ] , Y_train.iloc[val_idx]

        model = LGBMRegressor(
            n_estimators=1000,
            learning_rate=trial.suggest_float( 'learning_rate' , 0.01 , 0.1 ,),
            num_leaves = trial.suggest_int( 'num_leaves' , 15 , 200 ),
            verbose=-1, random_state=42, n_jobs=-1,
        )

        model.fit(X_tr, Y_tr , eval_set=[ (X_val,Y_val) ] ,
                    callbacks=[ early_stopping( stopping_rounds=50, verbose=False ) ] )

        r2 = model.score(X_val , Y_val)
        cv_r2_scores.append(r2)
        best_iterations.append(model.best_iteration_)

    trial.set_user_attr( 'best_n_estimators' , int( np.mean(best_iterations)))
    return np.mean( cv_r2_scores)

sampler = optuna.samplers.TPESampler( seed=42 )
study = optuna.create_study( direction='maximize' , sampler=sampler )
study.optimize( objective , n_trials=20 )

# 최적 모델
print( '최적 하이퍼파라미터:', study.best_params )
optimal_model = study.best_trial.user_attrs['best_n_estimators']
print( '최적 트리 개수: ', optimal_model )

# 최적 파라미터로 모델 학습 , 조기종료없이 최적으로
final_model = LGBMRegressor(
    random_state=42, verbose=-1, n_jobs=-1,
    **study.best_params,
    n_estimators=optimal_model
)
final_model.fit(X_train, Y_train)
print( 'train r2(log): ', final_model.score(X_train, Y_train))
print( 'test r2(log): ', final_model.score(X_test, Y_test))
from sklearn.metrics import r2_score
train_r2 = r2_score( np.expm1(Y_train) , np.expm1( final_model.predict(X_train) ) )
test_r2 = r2_score( np.expm1(Y_test) , np.expm1( final_model.predict(X_test) ) )
print( 'train r2(원): ', train_r2 )
print( 'test r2(원): ', test_r2 )
print( '차이(과적합): ', train_r2 - test_r2 )
print( final_model.feature_name_ )
print( final_model.feature_importances_ )




최적 하이퍼파라미터: {'learning_rate': 0.07158200161303108, 'num_leaves': 191}
최적 트리 개수:  886
train r2(log):  0.95202539266864
test r2(log):  0.916769514728996
train r2(원):  0.9661294919784472
test r2(원):  0.9452383631725133
차이(과적합):  0.020891128805933912
['전용면적', '층', '시군구', '법정동']
[106997  38097   6669  16577]


In [14]:
import joblib

metrics = {
    'test_r2'     : round( test_r2 , 4 ) ,      
    'train_r2'    : round( train_r2 , 4 ) ,
    'best_params' : study.best_params ,        
}
joblib.dump( final_model , f'{MODEL_DIR}/model_{sido}.pkl' )
joblib.dump( metrics , f'{MODEL_DIR}/metrics_{sido}.pkl' )
print( '저장 완료 :' , sido , metrics )

model = joblib.load( f'{MODEL_DIR}/model_{sido}.pkl' ) 

new_house = pd.DataFrame([
    {'전용면적': 84.9, '층': 10, '시군구': '성남분당구', '법정동': '정자동'},
    {'전용면적': 59.8, '층': 3,  '시군구': '수원영통구', '법정동': '영통동'},
])

for col in cat_cols:
    new_house[col] = new_house[col].astype('category')
new_house = new_house[num_cols + cat_cols]
preds = model.predict( new_house )
print( '서빙 모델 예측 (1번 집) 공시가격: ' , round( np.expm1( preds[0] ) ) , '원' )
print( '서빙 모델 예측 (2번 집) 공시가격: ' , round( np.expm1( preds[1] ) ) , '원' )

import shutil, os
DRIVE_MODEL_DIR = '/content/drive/MyDrive/house/models'
os.makedirs( DRIVE_MODEL_DIR , exist_ok=True )
for f in os.listdir( MODEL_DIR ):
    shutil.copy( f'{MODEL_DIR}/{f}' , DRIVE_MODEL_DIR )
print( '드라이브 저장 :' , os.listdir( DRIVE_MODEL_DIR ) )

저장 완료 : 경기도 {'test_r2': 0.9452, 'train_r2': 0.9661, 'best_params': {'learning_rate': 0.07158200161303108, 'num_leaves': 191}}
서빙 모델 예측 (1번 집) 공시가격:  1138424404 원
서빙 모델 예측 (2번 집) 공시가격:  243221284 원
드라이브 저장 : ['options.json', 'metrics_서울특별시.pkl', 'model_서울특별시.pkl', 'metrics_경기도.pkl', 'model_경기도.pkl']


In [15]:
sido = '서울특별시'
df_sido = df[ df['시도'] == sido ].sample(n=150_000, random_state=42)

num_cols = ['전용면적' , '층']
cat_cols = ['시군구' , '법정동']

for col in cat_cols:
    df_sido[col] = df_sido[col].astype('category')
X = df_sido[num_cols + cat_cols]

import numpy as np

Y = np.log1p( df_sido['공시가격'] )
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

import optuna
from lightgbm import LGBMRegressor, early_stopping

optuna.logging.set_verbosity( optuna.logging.WARNING )

# 목적 함수
def objective(trial) :
    from sklearn.model_selection import KFold
    cv = KFold( n_splits=3, shuffle=True, random_state=42 )
    cv_r2_scores = [ ]      # 각 폴드별 검증 점수 저장
    best_iterations = [ ]   # 각 폴드별 치적 트리 번호 저장
    for train_idx, val_idx in cv.split( X_train, Y_train) :
        X_tr, X_val = X_train.iloc[train_idx] , X_train.iloc[val_idx]
        Y_tr, Y_val = Y_train.iloc[ train_idx ] , Y_train.iloc[val_idx]

        model = LGBMRegressor(
            n_estimators=1000,
            learning_rate=trial.suggest_float( 'learning_rate' , 0.01 , 0.1 ,),
            num_leaves = trial.suggest_int( 'num_leaves' , 15 , 200 ),
            verbose=-1, random_state=42, n_jobs=-1,
        )

        model.fit(X_tr, Y_tr , eval_set=[ (X_val,Y_val) ] ,
                    callbacks=[ early_stopping( stopping_rounds=50, verbose=False ) ] )

        r2 = model.score(X_val , Y_val)
        cv_r2_scores.append(r2)
        best_iterations.append(model.best_iteration_)

    trial.set_user_attr( 'best_n_estimators' , int( np.mean(best_iterations)))
    return np.mean( cv_r2_scores)

sampler = optuna.samplers.TPESampler( seed=42 )
study = optuna.create_study( direction='maximize' , sampler=sampler )
study.optimize( objective , n_trials=20 )

# 최적 모델
print( '최적 하이퍼파라미터:', study.best_params )
optimal_model = study.best_trial.user_attrs['best_n_estimators']
print( '최적 트리 개수: ', optimal_model )

# 최적 파라미터로 모델 학습 , 조기종료없이 최적으로
final_model = LGBMRegressor(
    random_state=42, verbose=-1, n_jobs=-1,
    **study.best_params,
    n_estimators=optimal_model
)
final_model.fit(X_train, Y_train)
print( 'train r2(log): ', final_model.score(X_train, Y_train))
print( 'test r2(log): ', final_model.score(X_test, Y_test))
from sklearn.metrics import r2_score
train_r2 = r2_score( np.expm1(Y_train) , np.expm1( final_model.predict(X_train) ) )
test_r2 = r2_score( np.expm1(Y_test) , np.expm1( final_model.predict(X_test) ) )
print( 'train r2(원): ', train_r2 )
print( 'test r2(원): ', test_r2 )
print( '차이(과적합): ', train_r2 - test_r2 )
print( final_model.feature_name_ )
print( final_model.feature_importances_ )

최적 하이퍼파라미터: {'learning_rate': 0.04416438190782013, 'num_leaves': 179}
최적 트리 개수:  571
train r2(log):  0.9124939715567094
test r2(log):  0.8902119187094591
train r2(원):  0.8952233638993995
test r2(원):  0.875596722095728
차이(과적합):  0.019626641803671463
['전용면적', '층', '시군구', '법정동']
[60748 22854  3755 14281]
